In [52]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import pandas as pd
from backtesting.test import SMA, GOOG

In [ ]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import pandas as pd
from backtesting.test import SMA, GOOG

In [ ]:
#读取文件
bu = pd.read_csv('../data/bu.csv')
jd = pd.read_csv('../data/jd.csv')
l = pd.read_csv('../data/l.csv')
pp = pd.read_csv('../data/pp.csv')
ru = pd.read_csv('../data/ru.csv')
v = pd.read_csv('../data/v.csv')
bu

,date,open,high,low,close,volume,hold,settle
0,2015-06-26,2948.0,2972.0,2936.0,2960.0,32378,50834,2954.0
1,2015-06-29,2948.0,2970.0,2922.0,2936.0,31124,49566,2952.0
2,2015-06-30,2930.0,2948.0,2804.0,2812.0,52186,48312,2854.0
3,2015-07-01,2832.0,2850.0,2808.0,2826.0,31576,49450,2834.0
4,2015-07-02,2826.0,2830.0,2774.0,2816.0,36706,50466,2802.0
...,...,...,...,...,...,...,...,...
2427,2025-06-20,3738.0,3789.0,3723.0,3747.0,253286,284513,3751.0
2428,2025-06-23,3736.0,3804.0,3719.0,3781.0,317501,311332,3769.0
2429,2025-06-24,3762.0,3770.0,3562.0,3580.0,460577,269578,3643.0
2430,2025-06-25,3565.0,3583.0,3537.0,3574.0,282213,251579,3556.0


In [54]:
#重命名各列
def column_rename(df):
    df=df.rename(columns={
        df.columns[0]: 'Date',
        df.columns[1]: 'Open',
        df.columns[2]: 'High',
        df.columns[3]: 'Low',
        df.columns[4]: 'Close',
        df.columns[5]: 'Volume'
    }).drop(columns=[df.columns[6], df.columns[7]])
    df['Date'] = pd.to_datetime(df['Date'])
    return df.set_index('Date')
    
new_bu = column_rename(bu)

new_jd = column_rename(jd)
new_l = column_rename(l)
new_pp = column_rename(pp)
new_ru = column_rename(ru)
new_v = column_rename(v)
print(new_jd.head())
GOOG.head()

              Open    High     Low   Close  Volume
Date                                              
2015-06-26  4145.0  4145.0  4073.0  4085.0   76016
2015-06-29  4065.0  4090.0  4045.0  4068.0   70708
2015-06-30  4060.0  4060.0  3965.0  3969.0   96574
2015-07-01  4000.0  4112.0  3992.0  4028.0   69972
2015-07-02  4026.0  4029.0  3983.0  4009.0   54432


,Open,High,Low,Close,Volume
2004-08-19,100.00,104.06,95.96,100.34,22351900
2004-08-20,101.01,109.08,100.50,108.31,11428600
2004-08-23,110.75,113.48,109.05,109.40,9137200
2004-08-24,111.24,111.60,103.57,104.87,7631300
2004-08-25,104.96,108.00,103.88,106.00,4598900


In [55]:
# 纯双均值策略（0）
class SmaCross(Strategy):
    def init(self):
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)

    def next(self):
        if crossover(self.ma1, self.ma2):
            self.buy()
        elif crossover(self.ma2, self.ma1):
            self.sell()

In [56]:
#ATR计算函数
def ATR(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = abs(df['High'] - df['Close'].shift())
    low_close = abs(df['Low'] - df['Close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()


In [57]:
# 止损卖出（2）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        
        self.entry_price = 0
        self.stop_loss = 0

    def next(self):
        current_close = self.data.Close[-1]
        
        # 修改买入条件：添加持仓状态检查
        if self.entry_price ==0:
            if crossover(self.ma1, self.ma2) :
                self.buy()
                self.entry_price = current_close
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier
        
        # 强化卖出条件：添加持仓状态检查
        elif self.entry_price != 0:
            # 动态更新止损位
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            
            if crossover(self.ma2, self.ma1) or current_close < self.stop_loss:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0    # 重置止损位

In [58]:
# 尝试提前操作（3）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma0 = self.I(SMA, price, 1)
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        self.entry_price = 0
        self.stop_loss = 0
        self.current_date = self.data.df.index[0]
        self.cross1 = self.current_date
        self.cross2 = self.current_date
        self.fall1 = self.current_date
        self.fall2 = self.current_date

    def next(self):
        current_close = self.data.Close[-1]
        self.current_date = self.data.df.index[-1]
        self.cross1 = self.current_date if crossover(self.ma0, self.ma1) else self.cross1
        self.cross2 = self.current_date if crossover(self.ma0, self.ma2) else self.cross2
        self.fall1 = self.current_date if crossover(self.ma1, self.ma0) else self.fall1
        self.fall2 = self.current_date if crossover(self.ma2, self.ma0) else self.fall2
        below_ma50 = self.data.Close[-1] < self.filter[-1]
        if self.entry_price ==0:
            buy = False
            if self.cross2==self.current_date and below_ma50 and max(self.cross1,self.fall1,self.fall2)==self.cross1 and below_ma50:
                buy = True
            if buy:
                self.buy()
                self.entry_price = current_close  # 记录入场价格
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier  # 初始止损位
        
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            if 0.95 * self.entry_price >= current_close:
                sell = True
            if current_close < self.stop_loss :
                sell = True
            elif self.fall2==self.current_date and max(self.cross1,self.fall1,self.cross2)==self.fall1 and not below_ma50:
                sell = True
            if sell:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0  # 重置止损位


In [59]:
# 避免高位买入（4）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma0 = self.I(SMA, price, 1)
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        self.entry_price = 0
        self.sell_price = price* 1.5
        self.stop_loss = 0
        self.current_date = self.data.df.index[0]
        self.cross1 = self.current_date
        self.cross2 = self.current_date
        self.fall1 = self.current_date
        self.fall2 = self.current_date

    def next(self):
        current_close = self.data.Close[-1]
        self.current_date = self.data.df.index[-1]
        self.cross1 = self.current_date if crossover(self.ma0, self.ma1) else self.cross1
        self.cross2 = self.current_date if crossover(self.ma0, self.ma2) else self.cross2
        self.fall1 = self.current_date if crossover(self.ma1, self.ma0) else self.fall1
        self.fall2 = self.current_date if crossover(self.ma2, self.ma0) else self.fall2
        below_ma50 = self.data.Close[-1] < self.filter[-1]
        if self.entry_price ==0:
            buy = False
            self.sell_price = 1.01 * self.sell_price
            if self.cross2==self.current_date and below_ma50 and max(self.cross1,self.fall1,self.fall2)==self.cross1 and below_ma50:
                if self.sell_price != 0 and self.sell_price > current_close:
                    buy = True
            if buy:
                self.buy()
                self.entry_price = current_close  # 记录入场价格
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier  # 初始止损位
                self.sell_price = 0  # 重置卖出价格
        
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            if 0.95 * self.entry_price >= current_close:
                sell = True
            if current_close < self.stop_loss :
                sell = True
            elif self.fall2==self.current_date and max(self.cross1,self.fall1,self.cross2)==self.fall1 and not below_ma50:
                sell = True
            if sell:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0  # 重置止损位
                self.sell_price = current_close  # 记录卖出价格


In [60]:
# 尝试抓住迅猛上涨的行情（5）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma0 = self.I(SMA, price, 1)
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        self.entry_price = 0
        self.sell_price = price* 1.5
        self.stop_loss = 0
        self.current_date = self.data.df.index[0]
        self.cross1 = self.current_date
        self.cross2 = self.current_date
        self.fall1 = self.current_date
        self.fall2 = self.current_date

    def next(self):
        current_close = self.data.Close[-1]
        self.current_date = self.data.df.index[-1]
        self.cross1 = self.current_date if crossover(self.ma0, self.ma1) else self.cross1
        self.cross2 = self.current_date if crossover(self.ma0, self.ma2) else self.cross2
        self.fall1 = self.current_date if crossover(self.ma1, self.ma0) else self.fall1
        self.fall2 = self.current_date if crossover(self.ma2, self.ma0) else self.fall2
        below_ma50 = self.data.Close[-1] < self.filter[-1]
        if self.entry_price ==0:
            buy = False
            self.sell_price = 1.01 * self.sell_price
            if self.cross2==self.current_date and max(self.cross1,self.fall1,self.fall2)==self.cross1 and self.sell_price > current_close:
                if below_ma50:
                    buy = True
            if self.ma0[-1] > self.ma1[-1] and self.ma1[-1] > self.ma2[-1] and self.ma2[-1] > self.filter[-1]:
                buy = True
            if buy:
                self.buy()
                self.entry_price = current_close  # 记录入场价格
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier  # 初始止损位
                self.sell_price = 0  # 重置卖出价格
        
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            if 0.95 * self.entry_price >= current_close:
                sell = True
            if current_close < self.stop_loss :
                sell = True
            elif self.fall2==self.current_date and max(self.cross1,self.fall1,self.cross2)==self.fall1 and not below_ma50:
                sell = True
            if sell:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0  # 重置止损位
                self.sell_price = current_close  # 记录卖出价格


In [61]:
bt = Backtest(new_bu, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats


Backtest.run:   0%|          | 0/2382 [00:00<?, ?bar/s]

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    97.49178
Equity Final [$]                    37627.942
Equity Peak [$]                    166980.964
Commissions [$]                     47636.358
Return [%]                          -62.37206
Buy & Hold Return [%]                52.78731
Return (Ann.) [%]                    -9.63192
Volatility (Ann.) [%]                26.91668
CAGR [%]                              -6.5204
Sharpe Ratio                         -0.35784
Sortino Ratio                        -0.46952
Calmar Ratio                         -0.12434
Alpha [%]                           -65.40306
Beta                                  0.05742
Max. Drawdown [%]                   -77.46573
Avg. Drawdown [%]                    -9.61397
Max. Drawdown Duration     1914 days 00:00:00
Avg. Drawdown Duration      142 days 00:00:00
# Trades                          

In [62]:
bt = Backtest(new_jd, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Backtest.run:   0%|          | 0/2382 [00:00<?, ?bar/s]

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    94.40789
Equity Final [$]                     9841.732
Equity Peak [$]                    105654.948
Commissions [$]                     14816.248
Return [%]                          -90.15827
Buy & Hold Return [%]                -9.06762
Return (Ann.) [%]                   -21.35635
Volatility (Ann.) [%]                25.12821
CAGR [%]                            -14.78076
Sharpe Ratio                          -0.8499
Sortino Ratio                        -0.89631
Calmar Ratio                         -0.23226
Alpha [%]                           -91.88903
Beta                                 -0.19087
Max. Drawdown [%]                   -91.94952
Avg. Drawdown [%]                    -39.4043
Max. Drawdown Duration     3353 days 00:00:00
Avg. Drawdown Duration     1173 days 00:00:00
# Trades                          

In [63]:
bt = Backtest(new_l, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    96.54463
Equity Final [$]                     32728.75
Equity Peak [$]                     125170.77
Commissions [$]                      32042.41
Return [%]                          -67.27125
Buy & Hold Return [%]               -14.51991
Return (Ann.) [%]                   -10.93295
Volatility (Ann.) [%]                16.34351
CAGR [%]                             -7.41563
Sharpe Ratio                         -0.66895
Sortino Ratio                        -0.84096
Calmar Ratio                         -0.14589
Alpha [%]                           -66.80526
Beta                                  0.03209
Max. Drawdown [%]                   -74.93924
Avg. Drawdown [%]                   -10.04559
Max. Drawdown Duration     3149 days 00:00:00
Avg. Drawdown Duration      255 days 00:00:00
# Trades                          

In [64]:
bt = Backtest(new_pp, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Backtest.run:   0%|          | 0/2381 [00:00<?, ?bar/s]

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    93.78856
Equity Final [$]                    59019.342
Equity Peak [$]                    156286.742
Commissions [$]                     46402.666
Return [%]                          -40.98066
Buy & Hold Return [%]                -6.59658
Return (Ann.) [%]                    -5.31939
Volatility (Ann.) [%]                18.67438
CAGR [%]                             -3.57222
Sharpe Ratio                         -0.28485
Sortino Ratio                        -0.37668
Calmar Ratio                         -0.08386
Alpha [%]                           -39.50223
Beta                                  0.22412
Max. Drawdown [%]                   -63.42918
Avg. Drawdown [%]                    -9.51477
Max. Drawdown Duration     2109 days 00:00:00
Avg. Drawdown Duration      216 days 00:00:00
# Trades                          

In [65]:
bt = Backtest(new_ru, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    97.61513
Equity Final [$]                    129004.06
Equity Peak [$]                     266621.29
Commissions [$]                      73643.13
Return [%]                           29.00406
Buy & Hold Return [%]                37.91749
Return (Ann.) [%]                     2.67402
Volatility (Ann.) [%]                28.60804
CAGR [%]                              1.77237
Sharpe Ratio                          0.09347
Sortino Ratio                         0.15016
Calmar Ratio                          0.05057
Alpha [%]                            26.00177
Beta                                  0.07918
Max. Drawdown [%]                   -52.88108
Avg. Drawdown [%]                    -7.89274
Max. Drawdown Duration     1702 days 00:00:00
Avg. Drawdown Duration      132 days 00:00:00
# Trades                          

In [66]:
bt = Backtest(new_v, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Backtest.run:   0%|          | 0/192 [00:00<?, ?bar/s]

Start                     2021-09-15 00:00:00
End                       2022-09-15 00:00:00
Duration                    365 days 00:00:00
Exposure Time [%]                    59.91736
Equity Final [$]                   107092.128
Equity Peak [$]                     116576.34
Commissions [$]                      2253.376
Return [%]                            7.09213
Buy & Hold Return [%]               -20.69393
Return (Ann.) [%]                     7.39578
Volatility (Ann.) [%]                 22.4023
CAGR [%]                              4.84433
Sharpe Ratio                          0.33013
Sortino Ratio                         0.54291
Calmar Ratio                          0.48675
Alpha [%]                             8.11888
Beta                                  0.04962
Max. Drawdown [%]                   -15.19435
Avg. Drawdown [%]                    -4.54095
Max. Drawdown Duration      152 days 00:00:00
Avg. Drawdown Duration       33 days 00:00:00
# Trades                          